# Week 3 · Lesson 2 — Build and Trace a Mini-Transformer Encoder

You will assemble **two encoder layers** from multi-head self-attention (MHSA), residual additions, LayerNorm, and a feed-forward network (FFN). You will trace their shapes, inspect attention maps, and run three controlled ablations: **no positional encoding, no FFN contribution, and no residual connections**.

You construct the encoder yourself using PyTorch primitives, including `nn.MultiheadAttention`, as specified in your reading. You do not use `nn.TransformerEncoder`. To see inside MHSA, you will also reconstruct its Q/K/V projections and scaled dot-product attention from tensor operations and verify the result.

**Learning outcomes:** You should be able to explain how information flows through the two layers; verify shape preservation and attention normalization; distinguish token mixing from feature transformation; and support your ablation comments with numerical evidence.

All code is complete. Run the cells in order, then write your own **200–250 word reflection** in the final markdown cell. A small, supplied word-order task makes “degraded behavior” measurable; a large output change alone is not a quality measure.


## Step 1 — Setup and utilities
CPU is sufficient; a Colab GPU is optional. If packages are missing, the cell installs them. No dataset, account, pretrained model, or API key is required.

The main trace uses **B = 2, S = 6, d_model = 128**, with four heads. Each head has 32 features. All sub-block **outputs** preserve `(B, S, d_model)`; internal tensors have other shapes, including `(B, heads, S, S)` for attention and `(B, S, d_ff)` inside the FFN.


In [ ]:
import importlib.util
import subprocess
import sys

missing = [package for module, package in (("torch", "torch"), ("matplotlib", "matplotlib"))
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import math
import random
import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(2, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE, SEQ_LEN = 2, 6
D_MODEL, N_HEADS = 128, 4
D_FF, N_LAYERS = 4 * D_MODEL, 2
WORDS = ["approve", "reject", "budget", "client", "order", "invoice", "report", "team",
         "stock", "market", "price", "office", "meeting", "plan", "staff", "cost"]
VOCAB_SIZE = len(WORDS)
assert D_MODEL > 0 and N_HEADS > 0 and D_MODEL % N_HEADS == 0
assert N_LAYERS == 2 and SEQ_LEN >= 3

def stats(name, tensor):
    values = tensor.detach().float()
    print(f"{name:>20} | shape={str(tuple(tensor.shape)):>20} | "
          f"min={values.min().item(): .3f} max={values.max().item(): .3f} mean={values.mean().item(): .3f}")

def rms(a, b):
    return (a - b).float().square().mean().sqrt().item()

print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "Device:", device)
print(f"Two layers | B={BATCH_SIZE} | S={SEQ_LEN} | d_model={D_MODEL} | heads={N_HEADS} | d_ff={D_FF}")


## Step 2 — Sinusoidal positional encoding
Token embeddings identify tokens; the position signal identifies where they occur. The sinusoidal signal is stored as a non-trainable buffer and added to each token vector. The result keeps the same shape.

The implementation also handles odd embedding dimensions and checks its sequence-length limit. In this teaching model, embeddings are added directly to the position signal; no extra embedding scaling is used.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        if d_model < 1 or max_len < 1:
            raise ValueError("d_model and max_len must be positive.")
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                                * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * frequencies)
        pe[:, 1::2] = torch.cos(position * frequencies[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        if x.ndim != 3 or x.size(-1) != self.pe.size(-1):
            raise ValueError("Expected input shape (B, S, d_model).")
        if x.size(1) > self.pe.size(1):
            raise ValueError("Sequence exceeds the positional encoding's max_len.")
        return x + self.pe[:, :x.size(1)].to(dtype=x.dtype)

odd_demo = SinusoidalPositionalEncoding(5)(torch.zeros(2, SEQ_LEN, 5))
assert odd_demo.shape == (2, SEQ_LEN, 5)
print("Position encoding preserves shape, including odd feature dimensions.")


## Step 3 — One encoder layer: MHSA → Add & Norm → FFN → Add & Norm
This is a **post-norm** encoder: you add the residual first, then apply LayerNorm, following the original encoder arrangement (Vaswani et al., 2017).

- **MHSA** mixes information across token positions. `batch_first=True` makes its input/output `(B, S, d_model)`.
- **Add & LayerNorm** combines the branch output with its input and normalizes each token across its feature dimension. Once LayerNorm's scale and bias are trained, its output need not have an exact zero mean or unit variance.
- **FFN** applies the same learned nonlinear feature transformation independently to every token: `d_model → d_ff → d_model`. It does not attend to other positions.

Dropout is zero in this short diagnostic so repeat comparisons are deterministic and attention rows sum to one in evaluation mode. The two layers have separate parameters; stacking layers does not mean tying their weights.


In [ ]:
class MiniEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.0)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, trace=False, disable_ffn=False, disable_residuals=False):
        attn_out, weights = self.mha(x, x, x, need_weights=trace, average_attn_weights=False)
        attn_add = attn_out if disable_residuals else x + attn_out
        y = self.norm1(attn_add)
        if disable_ffn:
            expanded, ff = None, torch.zeros_like(y)
        elif trace:
            expanded = self.ffn[0](y)
            ff = self.ffn[2](self.ffn[1](expanded))
        else:
            expanded, ff = None, self.ffn(y)
        ff_add = ff if disable_residuals else y + ff
        z = self.norm2(ff_add)
        details = None
        if trace:
            # Reconstruct the MHSA internals from the SAME projection parameters.
            q, k, v = F.linear(x, self.mha.in_proj_weight, self.mha.in_proj_bias).chunk(3, dim=-1)
            split = lambda t: t.reshape(x.size(0), x.size(1), self.n_heads, self.head_dim).transpose(1, 2)
            q, k, v = split(q), split(k), split(v)
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            manual_weights = scores.softmax(dim=-1)
            combined = (manual_weights @ v).transpose(1, 2).contiguous().reshape_as(x)
            manual_out = self.mha.out_proj(combined)
            assert torch.allclose(weights, manual_weights, atol=2e-5, rtol=1e-4)
            assert torch.allclose(attn_out, manual_out, atol=2e-5, rtol=1e-4)
            details = {"input": x, "q_heads": q, "k_heads": k, "v_heads": v,
                "scores": scores, "attention": weights, "mhsa_out": attn_out,
                "attn_add": attn_add, "post_attn_norm": y, "ffn_hidden": expanded,
                "ffn_out": ff, "ffn_add": ff_add, "post_ffn_norm": z}
        return z, details


## Step 4 — Stack two layers and supply a small diagnostic task
You embed the IDs, add the position signal, and pass the result through two encoder layers. Ablations use flags on this **same encoder**; they never create fresh randomized models.

To interpret “degraded behavior,” you will briefly train a supplied diagnostic: **Does `approve` occur before `reject`?** Each six-token sequence contains both once, plus distractors. The words are labels in a synthetic task; this is not a sentiment or business-decision model.

For every sequence, a partner swaps only `approve` and `reject`, reversing the answer while preserving the same bag of tokens. Both train and test are balanced, and exact sequences do not overlap across sets. Mean pooling followed by a classifier converts the encoder's token outputs into one prediction. The task is deliberately sensitive to order; it is not a general NLP benchmark.

The training code is supplied so you can focus on architecture and evidence. A 50% majority-class baseline is appropriate here. Hardware and software versions may change the exact results.


In [ ]:
class MiniEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.posenc = SinusoidalPositionalEncoding(d_model)
        self.layers = nn.ModuleList([MiniEncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)])

    def forward(self, token_ids, trace=False, disable_position=False,
                disable_ffn=False, disable_residuals=False):
        embeddings = self.embed(token_ids)
        x = embeddings if disable_position else self.posenc(embeddings)
        input_trace = {"token_embeddings": embeddings, "encoder_input": x} if trace else None
        layer_traces = []
        for layer in self.layers:
            x, details = layer(x, trace=trace, disable_ffn=disable_ffn, disable_residuals=disable_residuals)
            if trace:
                layer_traces.append(details)
        return x, input_trace, layer_traces

class OrderClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(D_MODEL, 2)
    def forward(self, ids, **flags):
        outputs, _, _ = self.encoder(ids, **flags)
        return self.classifier(outputs.mean(dim=1))

def make_pairs(n, generator, seen):
    assert n % 2 == 0
    rows, labels = [], []
    while len(rows) < n:
        row = torch.randint(2, VOCAB_SIZE, (SEQ_LEN,), generator=generator)
        p, q = sorted(torch.randperm(SEQ_LEN, generator=generator)[:2].tolist())
        row[p], row[q] = 0, 1
        partner = row.clone()
        partner[p], partner[q] = 1, 0
        keys = (tuple(row.tolist()), tuple(partner.tolist()))
        if keys[0] in seen or keys[1] in seen:
            continue
        seen.update(keys)
        rows.extend((row, partner))
        labels.extend((1, 0))
    return torch.stack(rows), torch.tensor(labels, dtype=torch.long)

generator, seen = torch.Generator().manual_seed(SEED), set()
train_x, train_y = make_pairs(1024, generator, seen)
test_x, test_y = make_pairs(512, generator, seen)
assert set(map(tuple, train_x.tolist())).isdisjoint(set(map(tuple, test_x.tolist())))
assert train_y.sum().item() == len(train_y) // 2 and test_y.sum().item() == len(test_y) // 2
assert torch.equal(test_x[::2].sort(dim=1).values, test_x[1::2].sort(dim=1).values)
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), num_workers=0)
test_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=128, shuffle=False, num_workers=0)
torch.manual_seed(SEED)
encoder = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS).to(device)
probe = OrderClassifier(encoder).to(device)
assert encoder.layers[0].mha.in_proj_weight.data_ptr() != encoder.layers[1].mha.in_proj_weight.data_ptr()
print("Training examples:", len(train_x), "Held-out examples:", len(test_x))

@torch.no_grad()
def evaluate(**flags):
    probe.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for ids, labels in test_loader:
        ids, labels = ids.to(device), labels.to(device)
        logits = probe(ids, **flags)
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        correct += logits.argmax(dim=-1).eq(labels).sum().item()
        count += len(labels)
    return correct / count, loss_sum / count

EPOCHS = 12
optimizer = torch.optim.Adam(probe.parameters(), lr=0.002)
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    probe.train()
    loss_sum, count = 0.0, 0
    for ids, labels in train_loader:
        ids, labels = ids.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(probe(ids), labels)
        assert torch.isfinite(loss).item()
        loss.backward()
        nn.utils.clip_grad_norm_(probe.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item() * len(labels)
        count += len(labels)
    print(f"Diagnostic epoch {epoch:2}/{EPOCHS} | train loss={loss_sum/count:.4f}", flush=True)
baseline_accuracy, baseline_loss = evaluate()
print(f"Baseline held-out accuracy={baseline_accuracy:.1%}, loss={baseline_loss:.4f}; elapsed={time.perf_counter()-start:.1f}s")
print("Training is finished. All later ablations reuse these exact trained weights and this held-out set.")


## Step 5 — Trace shapes and verify the sub-blocks
You now trace two held-out sequences through the trained encoder. These partner sequences use the same tokens in different orders. You will print the embedding input, the Q/K/V head views, attention scores and weights, residual sums, FFN expansion, and both normalized outputs in **both layers**.

The assertions verify shape preservation, finite values, the actual residual additions, all heads' attention row sums, and agreement between the manual attention calculation and `nn.MultiheadAttention`. No padding or causal mask is used: these fixed-length encoder inputs attend bidirectionally to all six positions.


In [ ]:
token_ids = test_x[:BATCH_SIZE].to(device)
encoder.eval()
with torch.no_grad():
    enc_out, input_trace, traces = encoder(token_ids, trace=True)

print("=== INPUTS ===")
print("Token IDs shape:", tuple(token_ids.shape))
for row in token_ids.cpu().tolist():
    print("Tokens:", " | ".join(WORDS[idx] for idx in row))
for name, tensor in input_trace.items():
    stats(name, tensor)
expected = (len(token_ids), SEQ_LEN, D_MODEL)
assert len(traces) == N_LAYERS == 2 and enc_out.shape == expected
for layer_number, trace in enumerate(traces, start=1):
    print(f"\n=== LAYER {layer_number} ===")
    for name, tensor in trace.items():
        stats(name, tensor)
        assert torch.isfinite(tensor).all().item()
    for name in ("input", "mhsa_out", "attn_add", "post_attn_norm", "ffn_out", "ffn_add", "post_ffn_norm"):
        assert trace[name].shape == expected
    assert trace["ffn_hidden"].shape == (len(token_ids), SEQ_LEN, D_FF)
    assert trace["q_heads"].shape == (len(token_ids), N_HEADS, SEQ_LEN, D_MODEL // N_HEADS)
    assert trace["attention"].shape == (len(token_ids), N_HEADS, SEQ_LEN, SEQ_LEN)
    assert torch.allclose(trace["attn_add"], trace["input"] + trace["mhsa_out"])
    assert torch.allclose(trace["ffn_add"], trace["post_attn_norm"] + trace["ffn_out"])
    weights = trace["attention"]
    assert weights.min().item() >= 0
    assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=2e-5)
    print("Attention row sums, first head:", weights[0, 0].sum(dim=-1).cpu().tolist())
stats("encoder_output", enc_out)
print("\nAll shape, residual, attention-normalization, and manual-MHSA checks passed.")


## Step 6 — Visualize attention maps
Select a valid batch item and head. The plots show the same head in both layers, on the same 0–1 color scale.

**Rows are query positions; columns are key positions in the same input sequence.** Every row sums to one. Bright cells show how the head distributes attention for this input. They do not prove causal importance or human-like understanding; this model learned only the supplied synthetic ordering task.


In [ ]:
BATCH_INDEX, HEAD_INDEX = 0, 0
assert 0 <= BATCH_INDEX < len(token_ids) and 0 <= HEAD_INDEX < N_HEADS
labels = [f"{i}:{WORDS[idx]}" for i, idx in enumerate(token_ids[BATCH_INDEX].cpu().tolist())]
fig, axes = plt.subplots(1, N_LAYERS, figsize=(12, 5), constrained_layout=True)
for i, ax in enumerate(axes):
    matrix = traces[i]["attention"][BATCH_INDEX, HEAD_INDEX].cpu()
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(SEQ_LEN), labels, rotation=55, ha="right")
    ax.set_yticks(range(SEQ_LEN), labels)
    ax.set(title=f"Layer {i+1} · Head {HEAD_INDEX}", xlabel="Key position", ylabel="Query position")
fig.colorbar(image, ax=list(axes), label="Attention weight", shrink=0.8)
plt.show()


## Step 7 — Three controlled ablations
You change one component at a time, keeping **all trained weights, the classifier, inputs, test examples, evaluation mode, and pooling method identical**. There is no retraining and no fresh model initialization between conditions.

These are **inference-time ablations** of this trained model. They measure how it reacts to removing components it learned with. They do not establish how well a model without that component could perform if trained separately. Results on this task do not rank architectures for general NLP.

### Step 7a — Remove the positional signal
First compare baseline outputs with the same encoder with position addition bypassed. Then permute the input and compare corresponding token outputs after alignment. Without position information, this encoder is **permutation-equivariant**: reordering inputs reorders token outputs. With mean pooling, its classifier becomes permutation-invariant and cannot reliably distinguish the balanced partner sequences.

You must align corresponding output positions before comparing them. Attention maps must be permuted on **both** their query and key axes.


In [ ]:
# Keep a snapshot so we can confirm that evaluating ablations never changes weights.
saved_weights = {name: tensor.detach().clone() for name, tensor in probe.state_dict().items()}
encoder.eval()
permutation = torch.randperm(SEQ_LEN, generator=torch.Generator().manual_seed(7)).to(device)
with torch.no_grad():
    baseline, _, baseline_traces = encoder(token_ids, trace=True)
    no_position, _, no_position_traces = encoder(token_ids, trace=True, disable_position=True)
    permuted_no_position, _, permuted_traces = encoder(token_ids[:, permutation], trace=True, disable_position=True)
    permuted_with_position, _, _ = encoder(token_ids[:, permutation], trace=True)
    aligned_error = rms(permuted_no_position, no_position[:, permutation])
    aligned_error_with_position = rms(permuted_with_position, baseline[:, permutation])
assert torch.allclose(permuted_no_position, no_position[:, permutation], atol=2e-5, rtol=1e-4)
for original, permuted in zip(no_position_traces, permuted_traces):
    expected_attention = original["attention"][:, :, permutation, :][:, :, :, permutation]
    assert torch.allclose(permuted["attention"], expected_attention, atol=2e-5, rtol=1e-4)
print(f"Baseline vs no-position output RMS: {rms(baseline, no_position):.6f}")
print(f"Aligned permutation RMS, NO position: {aligned_error:.8f}")
print(f"Aligned permutation RMS, WITH position: {aligned_error_with_position:.8f}")
print("No-position token outputs and attention maps permute as expected.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, title, layer_traces in zip(axes, ("Baseline: with position", "Ablation: no position"),
                                   (baseline_traces, no_position_traces)):
    matrix = layer_traces[0]["attention"][BATCH_INDEX, HEAD_INDEX].cpu()
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(SEQ_LEN), labels, rotation=55, ha="right")
    ax.set_yticks(range(SEQ_LEN), labels)
    ax.set(title=title, xlabel="Key position", ylabel="Query position")
fig.colorbar(image, ax=list(axes), label="Attention weight", shrink=0.8)
plt.show()


### Step 7b — Remove the FFN contribution
You replace each FFN branch output with zeros, retaining the residual path and second LayerNorm. This isolates the contribution of the learned nonlinear FFN branch; it does not remove the whole second Add & Norm stage.

### Step 7c — Remove residual connections
You remove both residual additions in both layers, while retaining MHSA, the FFN, and LayerNorm. A large output difference alone does not show that performance is worse. Check the measured accuracy and loss below.


In [ ]:
conditions = {
    "Baseline": {},
    "No position": {"disable_position": True},
    "No FFN": {"disable_ffn": True},
    "No residuals": {"disable_residuals": True},
}
results = {}
with torch.no_grad():
    for name, flags in conditions.items():
        out, _, _ = encoder(token_ids, **flags)
        assert out.shape == baseline.shape and torch.isfinite(out).all().item()
        accuracy, loss = evaluate(**flags)
        results[name] = {"accuracy": accuracy, "loss": loss, "RMS": rms(baseline, out),
                         "drop_pp": 100 * (baseline_accuracy - accuracy)}

print(f"{'Condition':<16} {'Test accuracy':>14} {'Test loss':>11} {'Output RMS':>12} {'Accuracy drop (pp)':>20}")
for name, row in results.items():
    print(f"{name:<16} {row['accuracy']:>13.1%} {row['loss']:>11.4f} {row['RMS']:>12.6f} {row['drop_pp']:>20.2f}")
assert results["Baseline"]["RMS"] == 0.0
assert abs(results["Baseline"]["accuracy"] - baseline_accuracy) < 1e-12
assert all(torch.equal(tensor, saved_weights[name]) for name, tensor in probe.state_dict().items())
largest = max(("No FFN", "No residuals"), key=lambda name: results[name]["drop_pp"])
if results[largest]["drop_pp"] > 0:
    print(f"\nAmong FFN/residual removals, {largest} caused the larger observed accuracy drop on this task.")
else:
    print("\nNeither FFN nor residual removal reduced accuracy in this run; report that result honestly.")
print("All conditions preserve output shape. All model weights remain unchanged during ablations.")


## What your results can support
You have evidence about three different things:

| Evidence | What you may conclude |
|---|---|
| Shape traces, residual identities, attention row sums | The sub-blocks are wired consistently for these inputs. |
| Manual Q/K/V attention agrees with PyTorch | Your tensor calculation reproduces the MHSA operation with the same parameters. |
| Aligned permutation test | Without positional signals, token outputs and attention maps follow the input permutation. |
| Held-out diagnostic accuracy/loss | Which inference-time removal harmed this trained model's performance on this ordering task. |
| Output RMS | How much representations changed, not whether they improved or worsened. |

MHSA mixes information **between tokens**, whereas the FFN applies a learned nonlinear transformation **within each token's features**. Residual paths preserve an additive route for the incoming representation and help optimization when layers are trained. Your short inference-time experiment does not measure long-run gradient stability or prove that any component is unnecessary.

**Troubleshooting:** Ensure `D_MODEL % N_HEADS == 0`, retain `batch_first=True`, keep both residual inputs identically shaped, select a valid head and batch index, and verify that the baseline uses the positional layer. Attention can be diffuse; a flat map is not by itself a plotting error. If you change dimensions or training settings, restart and run all cells so the comparisons use one coherent model.


## Submit your notebook
Restart your kernel/runtime and run all cells in order. Submit one `.ipynb` containing:
1. All code cells executed, including Step 5 shape traces for both layers.
2. At least one Step 6 attention heat map (this notebook generates two), plus the positional comparison.
3. All three Step 7 ablation results and your own reflection below.

**References**
- Vaswani et al. (2017). *Attention Is All You Need.* [Paper](https://arxiv.org/abs/1706.03762).
- PyTorch. [MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html), [LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html).

You have now connected the encoder diagram to concrete tensors. In the next lesson, you will use these building blocks to distinguish encoder-only, decoder-only, and encoder–decoder models.


## Reflection — 200–250 words
Write your own response addressing all three questions. Cite printed values or observations from your run.

1. **Positional signal:** How did removing positional encoding in Step 7a affect the outputs or attention patterns? Refer to the aligned permutation test and/or the held-out ordering accuracy. Explain why changed token order requires aligning positions before comparison.
2. **FFN and residuals:** Which removal degraded behavior most in your run, and why might that be? Use held-out accuracy and/or loss to define degradation, and report the output RMS as a representation-change measure rather than a quality score. If neither removal degraded accuracy, or results tie, state that. Acknowledge that components were removed after training and that this is one synthetic task and seed.
3. **Complementary roles:** In one sentence, explain how MHSA and the FFN complement each other in the encoder.

**Your reflection:**

*Replace this line with your 200–250 word response. No sample answer is included.*
